# TreeShaker walkthrough: California Housing

End-to-end example on a real dataset (`sklearn.datasets.fetch_california_housing`).

We deliberately inject one **leaky** feature (a near-perfect proxy for the target) and one **redundant** feature (a duplicate of an existing column) to demonstrate TreeShaker's leakage and multicollinearity detection on top of normal importance ranking.

In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing

from treeshaker.config import Config
from treeshaker.pipeline import FeatureSelectionPipeline

data = fetch_california_housing(as_frame=True)
df = data.frame.copy()

rng = np.random.default_rng(42)

# Leaky feature: target plus tiny noise -- a model should latch onto this immediately.
df["price_leak"] = df["MedHouseVal"] + rng.normal(0, 0.01, size=len(df))

# Redundant feature: exact duplicate of AveRooms, should show up as multicollinear.
df["AveRooms_dup"] = df["AveRooms"]

df.shape, df.columns.tolist()

((20640, 11),
 ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude',
  'MedHouseVal',
  'price_leak',
  'AveRooms_dup'])

In [2]:
out_path = "california_housing_with_leaks.csv"
df.to_csv(out_path, index=False)

config = Config(
    input_file=out_path,
    target_column="MedHouseVal",
    task="regression",
    random_seed=42,
)
pipeline = FeatureSelectionPipeline(config)
results = pipeline.run()
results

,feature_name,shap_importance,permutation_importance,permutation_importance_std,mutual_info_score,recommendation,reason
0,price_leak,0.869935,2.558827e+00,1.322700e-02,4.435729,Caution,2 issues found — (1) Suspiciously high correla...
1,MedInc,0.030003,3.148941e-04,1.507635e-05,0.374530,Drop,Shuffling this feature barely changes model pe...
2,AveOccup,0.005425,6.383804e-05,5.226713e-06,0.066584,Drop,The model ignores this feature entirely — its ...
3,Latitude,0.004188,2.431635e-04,1.038307e-05,0.324520,Drop,2 issues found — (1) The model ignores this fe...
4,Longitude,0.003757,2.321830e-04,5.557932e-06,0.388589,Drop,2 issues found — (1) The model ignores this fe...
5,AveRooms,0.002153,2.108282e-05,1.615582e-06,0.107048,Drop,3 issues found — (1) The model ignores this fe...
6,HouseAge,0.001420,1.956400e-05,3.443010e-06,0.007767,Drop,The model ignores this feature entirely — its ...
7,AveRooms_dup,0.001063,1.049956e-05,8.285426e-07,0.106949,Drop,2 issues found — (1) The model ignores this fe...
8,AveBedrms,0.000344,2.666735e-06,4.173573e-07,0.000000,Drop,The model ignores this feature entirely — its ...
9,Population,0.000068,9.148833e-07,3.209895e-07,0.000000,Drop,The model ignores this feature entirely — its ...


## Reading the output

Running this notebook end to end produced the following (your exact numbers may vary slightly by environment/library versions, but the pattern is stable):

- `price_leak` gets flagged `Caution` with by far the highest SHAP/permutation importance, and the `reason` column calls out "Suspiciously high correlation" with the target -- exactly the leakage signal it should catch.
- Once `price_leak` dominates the model, the model leans on it so heavily that it stops needing the other features much -- they all get flagged `Drop` because "The model ignores this feature entirely." This is a realistic illustration of why leakage detection matters: a single leaky column can starve every other feature of importance, masking real signal (`MedInc` is normally the strongest predictor of California housing price, but it's nearly ignored here once `price_leak` is present).
- `AveRooms_dup`, the exact duplicate of `AveRooms`, also gets `Drop` -- both copies are individually low-importance once the model can use either one interchangeably, illustrating how multicollinearity dilutes importance across duplicated columns rather than always tripping a dedicated VIF flag.

In [3]:
results.sort_values(
    "shap_importance", ascending=False
) if "shap_importance" in results.columns else results

,feature_name,shap_importance,permutation_importance,permutation_importance_std,mutual_info_score,recommendation,reason
0,price_leak,0.869935,2.558827e+00,1.322700e-02,4.435729,Caution,2 issues found — (1) Suspiciously high correla...
1,MedInc,0.030003,3.148941e-04,1.507635e-05,0.374530,Drop,Shuffling this feature barely changes model pe...
2,AveOccup,0.005425,6.383804e-05,5.226713e-06,0.066584,Drop,The model ignores this feature entirely — its ...
3,Latitude,0.004188,2.431635e-04,1.038307e-05,0.324520,Drop,2 issues found — (1) The model ignores this fe...
4,Longitude,0.003757,2.321830e-04,5.557932e-06,0.388589,Drop,2 issues found — (1) The model ignores this fe...
5,AveRooms,0.002153,2.108282e-05,1.615582e-06,0.107048,Drop,3 issues found — (1) The model ignores this fe...
6,HouseAge,0.001420,1.956400e-05,3.443010e-06,0.007767,Drop,The model ignores this feature entirely — its ...
7,AveRooms_dup,0.001063,1.049956e-05,8.285426e-07,0.106949,Drop,2 issues found — (1) The model ignores this fe...
8,AveBedrms,0.000344,2.666735e-06,4.173573e-07,0.000000,Drop,The model ignores this feature entirely — its ...
9,Population,0.000068,9.148833e-07,3.209895e-07,0.000000,Drop,The model ignores this feature entirely — its ...


In [4]:
flagged = (
    results[results["recommendation"].isin(["Caution", "Drop"])]
    if "recommendation" in results.columns
    else None
)
flagged[
    ["feature_name", "recommendation", "reason"]
] if flagged is not None else flagged

,feature_name,recommendation,reason
0,price_leak,Caution,2 issues found — (1) Suspiciously high correla...
1,MedInc,Drop,Shuffling this feature barely changes model pe...
2,AveOccup,Drop,The model ignores this feature entirely — its ...
3,Latitude,Drop,2 issues found — (1) The model ignores this fe...
4,Longitude,Drop,2 issues found — (1) The model ignores this fe...
5,AveRooms,Drop,3 issues found — (1) The model ignores this fe...
6,HouseAge,Drop,The model ignores this feature entirely — its ...
7,AveRooms_dup,Drop,2 issues found — (1) The model ignores this fe...
8,AveBedrms,Drop,The model ignores this feature entirely — its ...
9,Population,Drop,The model ignores this feature entirely — its ...
